In [1]:
import sys
import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"
import json
import torch
import random
import warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')
import torch._dynamo
torch._dynamo.disable()
# Set seed
seed = 42
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Add src path
src_path = os.path.abspath(os.path.join('..', 'src2'))
sys.path.append(src_path)

# Device
device = os.getenv("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Imports
import active_gliner as ag
from active_gliner.create_data.gliner_format import convert_raw_json_to_gliner_training
from active_gliner.config.data_paths import (
    MIT_movies_NER_train_path,
    MIT_movies_NER_test_path,
    MIT_movies_NER_labels_path
)
from active_gliner.get_model.DefaultModel import DefaultModel
from active_gliner.evaluate_model.get_metrics import (
    evaluate_with_ground_truth,
    evaluate_without_ground_truth
)




Using device: cuda


2025-11-24 01:15:51.174051444 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card5/device/vendor"


In [2]:
from active_gliner.create_data.gliner_utils import analyze_entity_distribution

# Load data
print("\n" + "="*80)
print("LOADING DATA")
print("="*80)

with open(MIT_movies_NER_train_path, 'r') as f:
    train_data = json.load(f)

with open(MIT_movies_NER_test_path, 'r') as f:
    test_data = json.load(f)

with open(MIT_movies_NER_labels_path, 'r') as f:
    labels = json.load(f)

print(f"Train examples: {len(train_data)}")
print(f"Test examples: {len(test_data)}")
print(f"Entity types: {labels}")

# Convert to GLiNER format
converted_train_data = convert_raw_json_to_gliner_training(train_data)  
converted_test_data = convert_raw_json_to_gliner_training(test_data)

train_stats = analyze_entity_distribution(converted_train_data, "train")
test_stats = analyze_entity_distribution(converted_test_data, "test")


train_stats=pd.DataFrame(train_stats)
test_stats=pd.DataFrame(test_stats) 

distribution_df=pd.merge(train_stats,test_stats, on ='entity',how='inner')
display(distribution_df)


LOADING DATA
Train examples: 9774
Test examples: 2442
Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']

Entity Distribution Analysis: train
Total examples: 9,774
Total tokens: 99,479
Total entities: 21,294
Unique entity types: 12
Average entities per example: 2.18

Entity                    Count      Percentage
---------------------------------------------

Entity density: 21.41% of tokens are entities

Analysis completed in 0.00 seconds

Entity Distribution Analysis: test
Total examples: 2,442
Total tokens: 24,675
Total entities: 5,338
Unique entity types: 12
Average entities per example: 2.19

Entity                    Count      Percentage
---------------------------------------------

Entity density: 21.63% of tokens are entities

Analysis completed in 0.00 seconds


,entity,count_train,percentage_train,count_test,percentage_test
0,genre,4354,20.447074,1117,20.925440
1,actor,3220,15.121631,812,15.211690
2,year,2858,13.421621,720,13.488198
3,title,2376,11.158073,562,10.528288
4,rating,2007,9.425190,500,9.366804
5,plot,1927,9.049498,491,9.198202
6,average ratings,1869,8.777120,451,8.448857
7,director,1720,8.077393,456,8.542525
8,character,384,1.803325,89,1.667291
9,song,245,1.150559,54,1.011615


In [3]:
# Load model
model = DefaultModel(device=device)

print("\nLoading base model (no adapter found)")
model.load_for_inference()


Loading base model (no adapter found)
Loading GLiNER for inference: knowledgator/modern-gliner-bi-large-v1.0


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 45811.57it/s]


Using base model


# Predict with model

In [4]:
# Get predictions on test set
test_texts = [example['sentence'] for example in test_data]
print(f"Generating predictions for {len(test_texts)} examples...")
test_predictions = []
for text in test_texts:
    # Use flat_ner=False to match GLiNER's evaluate() behavior
    pred = model.predict_entities(text, labels, threshold=0.5, flat_ner=False)

    test_predictions.append(pred)

for i in test_predictions[:3]:
    print(i)

# Evaluate
test_results = evaluate_with_ground_truth(
    predictions=test_predictions,
    data=converted_test_data,
    entity_types=labels,
    has_confidence=True
)

Generating predictions for 2442 examples...
[{'start': 19, 'end': 36, 'text': 'romantic comedies', 'label': 'genre', 'score': 0.8240767121315002}]
[]
[{'start': 41, 'end': 51, 'text': 'mel gibson', 'label': 'actor', 'score': 0.9341018199920654}]


# Baseline Results

In [5]:
from active_gliner.evaluate_model.utils import display_results

display_results(test_results)


EVALUATION RESULTS

Overall Metrics:
Total Predictions: 3,932
Overall Confidence: 0.7742 (77.42%)
Total Examples: 2,442
Correct Examples: 315
Incorrect Examples: 2,127
Example-Level Accuracy: 0.1290 (12.90%)
Entity-Level Accuracy: 0.4046 (40.46%)
Overall F1 Score: 0.4660 (46.60%)

Confidence Distribution:


,entity_type,0-25%,26-50%,51-75%,76-100%
0,genre,0,0,550,503
1,year,0,0,88,200
2,plot,0,0,239,329
3,average ratings,0,0,4,5
4,actor,0,0,158,484
5,title,0,0,19,11
6,song,0,0,13,25
7,character,0,0,143,174
8,rating,0,0,219,360
9,review,0,0,49,36



Classification Report:


,entity_type,precision,recall,f1,tp,fp,fn,support,avg_confidence
0,genre,0.587844,0.554163,0.570507,619,434,498,1117,0.79
1,year,0.711806,0.284722,0.406746,205,83,515,720,0.88
2,plot,0.246479,0.285132,0.264400,140,428,351,491,0.78
3,average ratings,0.111111,0.002217,0.004348,1,8,450,451,0.54
4,actor,0.914330,0.722906,0.807428,587,55,225,812,0.83
5,title,0.066667,0.003559,0.006757,2,28,560,562,0.60
6,song,0.552632,0.388889,0.456522,21,17,33,54,0.86
7,character,0.217666,0.775281,0.339901,69,248,20,89,0.89
8,rating,0.374784,0.434000,0.402224,217,362,283,500,0.77
9,review,0.070588,0.107143,0.085106,6,79,50,56,0.81



True Positives Confidence Analysis:


,entity_type,0-25%,26-50%,51-75%,76-100%
0,genre,0,0,222,397
1,year,0,0,24,181
2,plot,0,0,65,75
3,average ratings,0,0,1,0
4,actor,0,0,138,449
5,title,0,0,2,0
6,song,0,0,4,17
7,character,0,0,11,58
8,rating,0,0,84,133
9,review,0,0,1,5



False Positives Confidence Analysis:


,entity_type,0-25%,26-50%,51-75%,76-100%
0,genre,0,0,328,106
1,year,0,0,64,19
2,plot,0,0,174,254
3,average ratings,0,0,3,5
4,actor,0,0,20,35
5,title,0,0,17,11
6,song,0,0,9,8
7,character,0,0,132,116
8,rating,0,0,135,227
9,review,0,0,48,31


# Finetune model

In [6]:
# Delete training model instance
import gc
del model  # or whatever your training model variable is called
torch.cuda.empty_cache()
gc.collect()

107

In [7]:


# Use  test data for eval during training (faster)
print(f"Using {len(converted_test_data)} examples for validation during training")

# Initialize DefaultModel
print("\nInitializing DefaultModel with LoRA...")
finetuned_model = DefaultModel()
finetuned_model.load_for_training()
print(f"Training model on device: {finetuned_model.device}")

# Set adapter save path
adapter_save_path = "../models/finetune_model_adapter"
os.makedirs(adapter_save_path, exist_ok=True)

# Fine-tune the model
print(f"\nStarting fine-tuning on {len(converted_train_data)} examples...")
print(f"This will take several minutes...\n")

finetuned_model.fit(
    train_data=converted_train_data,
    eval_data=converted_test_data,
    adapter_save_path=adapter_save_path
)

print("\n" + "="*80)
print("FINE-TUNED MODEL EVALUATION")
print("="*80)

# After training, before evaluation
import gc
import torch

# Delete training model instance
del finetuned_model  # or whatever your training model variable is called
torch.cuda.empty_cache()
gc.collect()

Using 2442 examples for validation during training

Initializing DefaultModel with LoRA...
Loading GLiNER for training: knowledgator/modern-gliner-bi-large-v1.0


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 167029.81it/s]


Applying training LoRA: r=16, alpha=32
trainable params: 10,407,936 || all params: 540,253,184 || trainable%: 1.9265
Training model on device: cuda

Starting fine-tuning on 9774 examples...
This will take several minutes...

Training on 9774 examples
Eval on 2442 examples


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50283}.


Step,Training Loss,Validation Loss
100,8.164900,52.998764
200,5.523700,35.676132
300,4.600900,37.134277
400,3.806500,23.433214
500,5.141400,30.270588
600,4.148600,32.826939
700,4.228200,30.785789
800,4.538500,27.913807
900,4.314600,23.823999
1000,4.514300,40.295628


Skipping iteration due to error: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 0 has a total capacity of 5.67 GiB of which 6.69 MiB is free. Including non-PyTorch memory, this process has 5.64 GiB memory in use. Of the allocated memory 5.28 GiB is allocated by PyTorch, and 259.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Adapter saved to ../models/finetune_model_adapter

FINE-TUNED MODEL EVALUATION


15601

# Load finetuneed model

In [8]:
# Load model
adapter_path = "../models/finetune_model_adapter"
model = DefaultModel(device=device)

if os.path.exists(adapter_path):
    print(f"\nLoading fine-tuned model from {adapter_path}")
    model.load_for_inference(adapter_path=adapter_path)
else:
    print("\nLoading base model (no adapter found)")
    model.load_for_inference()

# Get predictions on test set
test_texts = [example['sentence'] for example in test_data]
print(f"Generating predictions for {len(test_texts)} examples...")
test_predictions = []
for text in test_texts:
    # Use flat_ner=False to match GLiNER's evaluate() behavior
    pred = model.predict_entities(text, labels, threshold=0.5, flat_ner=False)

    test_predictions.append(pred)

for i in test_predictions[:3]:
    print(i)

# Evaluate
test_results = evaluate_with_ground_truth(
    predictions=test_predictions,
    data=converted_test_data,
    entity_types=labels,
    has_confidence=True
)


display_results(test_results)


Loading fine-tuned model from ../models/finetune_model_adapter
Loading GLiNER for inference: knowledgator/modern-gliner-bi-large-v1.0


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 154076.47it/s]


Loading adapter: ../models/finetune_model_adapter
Active adapter: default
Generating predictions for 2442 examples...
[{'start': 14, 'end': 18, 'text': 'good', 'label': 'review', 'score': 0.7967699766159058}, {'start': 19, 'end': 36, 'text': 'romantic comedies', 'label': 'genre', 'score': 0.8619595170021057}]
[{'start': 22, 'end': 36, 'text': 'cars that talk', 'label': 'plot', 'score': 0.7689406871795654}]
[{'start': 9, 'end': 13, 'text': 'five', 'label': 'average ratings', 'score': 0.615433931350708}, {'start': 9, 'end': 18, 'text': 'five star', 'label': 'average ratings', 'score': 0.5311034917831421}, {'start': 41, 'end': 51, 'text': 'mel gibson', 'label': 'actor', 'score': 0.9242669343948364}]

EVALUATION RESULTS

Overall Metrics:
Total Predictions: 5,953
Overall Confidence: 0.8647 (86.47%)
Total Examples: 2,442
Correct Examples: 1,399
Incorrect Examples: 1,043
Example-Level Accuracy: 0.5729 (57.29%)
Entity-Level Accuracy: 0.8816 (88.16%)
Overall F1 Score: 0.8336 (83.36%)

Confidenc

,entity_type,0-25%,26-50%,51-75%,76-100%
0,genre,0,0,93,1077
1,year,0,0,81,739
2,plot,0,0,121,362
3,average ratings,0,0,116,386
4,actor,0,0,75,768
5,title,0,0,235,470
6,song,0,0,27,52
7,character,0,0,40,102
8,rating,0,0,52,486
9,review,0,0,145,47



Classification Report:


,entity_type,precision,recall,f1,tp,fp,fn,support,avg_confidence
0,genre,0.899145,0.941808,0.919983,1052,118,65,1117,0.92
1,year,0.831707,0.947222,0.885714,682,138,38,720,0.93
2,plot,0.734990,0.723014,0.728953,355,128,136,491,0.84
3,average ratings,0.786853,0.875831,0.828961,395,107,56,451,0.89
4,actor,0.889680,0.923645,0.906344,750,93,62,812,0.89
5,title,0.690780,0.866548,0.768745,487,218,75,562,0.89
6,song,0.531646,0.777778,0.631579,42,37,12,54,0.89
7,character,0.471831,0.752809,0.580087,67,75,22,89,0.91
8,rating,0.795539,0.856000,0.824663,428,110,72,500,0.93
9,review,0.140625,0.482143,0.217742,27,165,29,56,0.69



True Positives Confidence Analysis:


,entity_type,0-25%,26-50%,51-75%,76-100%
0,genre,0,0,34,1018
1,year,0,0,16,666
2,plot,0,0,54,301
3,average ratings,0,0,42,353
4,actor,0,0,39,711
5,title,0,0,73,414
6,song,0,0,4,38
7,character,0,0,4,63
8,rating,0,0,20,408
9,review,0,0,17,10



False Positives Confidence Analysis:


,entity_type,0-25%,26-50%,51-75%,76-100%
0,genre,0,0,59,59
1,year,0,0,65,73
2,plot,0,0,67,61
3,average ratings,0,0,74,33
4,actor,0,0,36,57
5,title,0,0,162,56
6,song,0,0,23,14
7,character,0,0,36,39
8,rating,0,0,32,78
9,review,0,0,128,37
